# Profit vs Pay: UK Energy and Utilities

This notebook is a short walkthrough of the project. The aim is simple: compare how CEO pay and median employee pay have changed over time inside major UK energy and utilities companies.

The raw source is company pay-ratio disclosure data. I cleaned the dataset, narrowed it to energy-related firms, kept only companies with enough history to compare over time, and then re-indexed CEO pay and median employee pay to a common starting point of `100`.

In [ ]:
import pandas as pd

wages = pd.read_csv('data/cleaned/wages.csv')
companies = pd.read_csv('data/cleaned/companies.csv')

print('Companies:', wages['Company'].nunique())
print('Rows:', len(wages))
print('Coverage:', int(wages['Year'].min()), 'to', int(wages['Year'].max()))
companies

## Why index the series?

A CEO can earn far more than a median employee in absolute terms, but that alone does not tell you how fast the gap is changing. Re-indexing both lines to `100` in the first available year makes the trend easier to compare.

- A line ending at `150` means pay is `50%` above its starting point.
- A line ending at `80` means pay is `20%` below its starting point.
- The bigger the separation between the two lines, the more the company has pulled apart over time.

In [ ]:
summary_rows = []

for company, group in wages.groupby('Company'):
    group = group.sort_values('Year').copy()
    ceo_start = group['ceo_pay_k'].iloc[0] * 1000
    ceo_end = group['ceo_pay_k'].iloc[-1] * 1000
    median_start = group['m_pay'].iloc[0]
    median_end = group['m_pay'].iloc[-1]

    summary_rows.append(
        {
            'Company': company,
            'Start year': int(group['Year'].iloc[0]),
            'End year': int(group['Year'].iloc[-1]),
            'CEO change %': round((ceo_end / ceo_start - 1) * 100, 1),
            'Median change %': round((median_end / median_start - 1) * 100, 1),
            'Indexed gap (points)': round((ceo_end / ceo_start) * 100 - (median_end / median_start) * 100, 1),
        }
    )

summary = pd.DataFrame(summary_rows).sort_values('Indexed gap (points)', ascending=False)
summary

## A few takeaways

- Centrica is the sharpest divergence in the current sample: CEO pay is up by roughly `264%` from 2019 to 2024, while median pay is up by roughly `35%`.
- Drax also shows a strong split, with CEO pay growth well ahead of worker pay growth.
- The pattern is mixed rather than universal. BP and Royal Dutch Shell show cases where indexed median pay rose while indexed CEO pay ended below its own starting level.

That makes the project more interesting than a simple one-direction story. The dataset exposes both divergence and restraint, depending on the company.

## Example charts

### BP

![BP chart](data/cleaned/charts/bp_indexed_pay.png)

### Centrica

![Centrica chart](data/cleaned/charts/centrica_indexed_pay.png)

### Drax

![Drax chart](data/cleaned/charts/drax_indexed_pay.png)

## Rebuild steps

If I want to refresh the cleaned data and charts, I run:

```python
python scripts/dataCleaner.py
python scripts/plotIndexedPayCharts.py
```

The cleaned outputs live in `data/cleaned/`, and the generated figures live in `data/cleaned/charts/`.